### Topic 5 - Qdrant Flow

This example demonstrates Qdrant's core data model using **Collections**, **Points**, **Payloads**, **Deterministic IDs**, **Payload Indexes**, **Retrieve**, **Scroll**, and **Upsert**.

The program performs the following steps in order:

1. Create the Qdrant client.
2. Load the embedding model.
3. Create a collection.
4. Add a payload index.
5. Generate deterministic Point IDs.
6. Generate embeddings for all documents.
7. Create Qdrant Points.
8. Upsert Points into Qdrant.
9. Retrieve a Point by its ID.
10. Scroll through stored Points.
11. Demonstrate idempotent upsert.

---

### Step 1: Create Qdrant Client

```python
client = QdrantClient(":memory:")
```

**What Happens:** Creates an in-memory Qdrant database. No server or Docker is required. Data exists only while the Python program is running.

**Theory:** `:memory:` creates a temporary Qdrant instance. Everything is stored in RAM and is deleted when the program exits.

**Internally:** An empty Qdrant database is created. No collections, Points, payloads, or indexes exist yet.

---

### Step 2: Load the Embedding Model

```python
model = SentenceTransformer(MODEL_NAME)
```

**What Happens:** Loads the Sentence Transformer model into memory.

**Theory:** Qdrant stores and searches vectors but does not generate them. Embeddings are created externally using the embedding model. The same model must be used for both document and query embeddings.

**Example:** Every FD document, such as **"Can I withdraw my FD before maturity?"**, is converted into a 384-dimensional embedding before being stored in Qdrant.

**Internally:** The Transformer generates token embeddings, applies mean pooling to produce a single sentence embedding, and returns a 384-dimensional vector. Nothing is stored in Qdrant yet.

---

### Step 3: Create Collection

```python
create_collection()
```

Inside:

```python
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=VECTOR_SIZE,
        distance=Distance.COSINE,
    ),
)
```

**What Happens:** Creates a collection named **fd_knowledge_base** to store all FD-related vectors.

**Theory:** A **Collection** is the highest-level container in Qdrant, similar to a SQL table. It stores multiple Points and defines properties shared by every Point, including the vector dimension and similarity metric.

**Example:** The **fd_knowledge_base** collection stores document chunks from FD policies, product guides, and FAQs in a single searchable collection.

**Collection Configuration:**

- **Vector Size:** `384` (must match the embedding model output).
- **Distance Metric:** `Cosine Similarity`.
- Every Point stored in this collection must follow these settings.

**Internally:** Qdrant creates an empty collection and stores its configuration. No Points or vectors exist yet.

**Output:**

```text
Created collection 'fd_knowledge_base'
```

This confirms that the collection has been created successfully and is ready to store vectors.

---

### Step 4: Add Payload Index

```python
add_payload_index()
```

Inside:

```python
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="doc_type",
    field_schema=PayloadSchemaType.KEYWORD,
)
```

**What Happens:** Creates a payload index on the **`doc_type`** field to speed up metadata filtering.

**Theory:** A **Payload Index** is similar to an index in a SQL database. Without an index, Qdrant checks every Point's payload while applying a filter. With an index, Qdrant can directly locate matching Points, making filtered searches much faster on large collections.

**Example:** Suppose the collection contains **1 million** Points.

Filter:

```python
doc_type = "faq"
```

- **Without Payload Index:** Qdrant checks the payload of every Point.
- **With Payload Index:** Qdrant directly locates Points where `doc_type="faq"`.

**Internally:** Qdrant creates an index for the `doc_type` field. As new Points are inserted, the index is automatically updated.

**Output:**

```text
Added payload index on 'doc_type'
```

This confirms that the payload index has been created successfully.

**Note:** The warning shown at the end of the output:

```text
Payload indexes have no effect in the local Qdrant.
```

appears because this demo uses **`:memory:`** mode. In a production Qdrant server, payload indexes significantly improve filtering performance.

---

### Step 5: Generate Deterministic Point IDs

```python
point_id = make_point_id(
    meta["source"],
    meta["chunk_index"]
)
```

Inside:

```python
key = f"{source}::{chunk_index}"

return int(
    hashlib.md5(key.encode()).hexdigest()[:8],
    16,
)
```

**What Happens:** Generates the same integer ID every time the same document chunk is processed.

**Theory:** Every Point in Qdrant requires a unique ID. Using a **deterministic ID** ensures that re-ingesting the same document updates the existing Point instead of creating a duplicate.

**Example:** For the document:

```text
Source      : FD_Policy.json
Chunk Index : 0
```

the generated ID is always:

```text
3267135047
```

If the document is ingested again with updated content, the same ID is generated, allowing Qdrant to update the existing Point.

**Why Not Random IDs?**

- Random IDs create duplicate Points during re-ingestion.
- Deterministic IDs make ingestion **idempotent**, meaning the same operation can be repeated without creating duplicates.

**Internally:** The source filename and chunk index are combined into a unique string, hashed using MD5, and converted into an integer. The same input always produces the same ID.

---

### Step 6: Generate Embeddings

```python
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
)
```

**What Happens:** Converts every document into a **384-dimensional normalized embedding**.

**Theory:** The Sentence Transformer uses a **Transformer encoder** to understand the semantic meaning of each document. It then applies **mean pooling** to generate a single sentence embedding. Setting `normalize_embeddings=True` converts each embedding into a **unit vector**, making **Cosine Similarity** efficient and consistent during search.

**Example:** The documents:

- **"Premature withdrawal incurs a 1 percent penalty..."**
- **"Can I withdraw my FD before maturity?"**

generate similar embeddings because they discuss the same FD withdrawal concept, even though the wording is different.

**Internally:** The embedding model processes every document, generates token embeddings, applies mean pooling, normalizes the resulting vectors, and returns one 384-dimensional embedding for each document. Qdrant is not involved in this step.

---

### Step 7: Create Qdrant Points

```python
PointStruct(...)
```

**What Happens:** Creates one Qdrant Point for every document.

**Theory:** A **Point** is the basic storage unit in Qdrant, similar to a row in a SQL table. Every Point contains:

- Unique ID
- Embedding Vector
- Payload (metadata)

The **Payload** stores information about the document but is **not** used to calculate semantic similarity. Instead, it is used for filtering, retrieval, and displaying search results.

**Example:** One FD FAQ is stored as:

- **ID:** `3267135047`
- **Vector:** 384-dimensional embedding
- **Payload:**
  - `text`
  - `source`
  - `doc_type`
  - `page`
  - `chunk_index`

When a user searches for **"Can I close my FD early?"**, the **vector** helps find similar documents, while the **payload** provides the original text, source file, and document type.

**Internally:** Python creates `PointStruct` objects in memory. They are not stored in Qdrant until `upsert()` is called.

---

### Step 8: Upsert Points

```python
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
)
```

**What Happens:** Stores all Points inside the collection. If a Point with the same ID already exists, it is updated instead of creating a duplicate.

**Theory:** **Upsert** combines **Insert** and **Update** into a single operation.

- **New ID:** Insert a new Point.
- **Existing ID:** Replace the existing Point.

This makes incremental ingestion simple and prevents duplicate vectors.

**Example:** The first time the document:

```text
Premature withdrawal incurs a 1 percent penalty.
```

is inserted, a new Point is created.

If the same document is later updated to:

```text
UPDATED: Premature withdrawal penalty is 1 percent.
```

the same deterministic ID is generated, so Qdrant replaces the existing Point instead of creating another one.

**Internally:** For every Point, Qdrant:

- Stores the ID.
- Stores the embedding vector.
- Stores the payload.
- Updates the payload index automatically.

**Output:**

```text
Upserted 5 points | Total in collection: 5
```

This confirms that all five documents were successfully inserted into the collection.

---

### Step 9: Retrieve a Point by ID

```python
retrieve_by_id(
    "FD_Policy.json",
    chunk_index=0,
)
```

Inside:

```python
results = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[point_id],
    with_payload=True,
    with_vectors=False,
)
```

**What Happens:** Retrieves a specific Point using its ID. No query embedding or semantic search is performed.

**Theory:** **Retrieve** is used when the Point ID is already known. Unlike semantic search, Qdrant directly fetches the matching Point instead of comparing vectors.

**Example:** The deterministic ID generated for:

- **Source:** `FD_Policy.json`
- **Chunk Index:** `0`

is used to directly retrieve the corresponding Point.

**Internally:** Qdrant looks up the Point by its ID and returns the requested fields.

- `with_payload=True` returns the metadata.
- `with_vectors=False` skips the embedding vector to reduce the response size.

**Output:**

```text
Retrieved point ID=3267135047:
  text     : Premature withdrawal incurs a 1 percent penalty on the applicable rate.
  doc_type : policy
  source   : FD_Policy.json
```

This confirms that the Point was retrieved successfully using only its ID.

---

### Step 10: Scroll Through Points

```python
scroll_all()
```

Inside:

```python
client.scroll(...)
```

**What Happens:** Iterates through stored Points without requiring a query embedding.

**Theory:** **Scroll** is used to browse Points stored in a collection. Unlike semantic search, it does not calculate vector similarity. It is commonly used for auditing, bulk updates, re-embedding, and data migration.

**Example:** The first call returns all stored documents:

```text
Scroll (all): 5 points
```

The second call applies a payload filter:

```python
doc_type = "faq"
```

and returns only FAQ documents.

**Internally:** Qdrant sequentially reads Points from the collection. When a filter is provided, only Points whose payload matches the filter are returned.

**Output:**

```text
Scroll (all): 5 points
```

Shows every stored Point.

```text
Scroll (doc_type='faq'): 2 points
```

Shows only the two FAQ documents because the payload filter matches `doc_type="faq"`.

---

### Step 11: Demonstrate Idempotent Upsert

```python
demonstrate_idempotent_upsert()
```

**What Happens:** Re-ingests an existing document using the same deterministic ID to demonstrate that the existing Point is updated instead of duplicated.

**Theory:** An **idempotent upsert** means repeating the same operation produces the same final state. Deterministic IDs ensure the existing Point is replaced rather than creating a duplicate.

**Example:** The original document:

```text
Premature withdrawal incurs a 1 percent penalty.
```

is re-ingested as:

```text
UPDATED: Premature withdrawal penalty is 1 percent.
```

Because both documents generate the same Point ID, Qdrant updates the existing Point.

**Internally:** Qdrant detects that the ID already exists, replaces the stored Point with the new one, and updates the payload index automatically.

**Output:**

```text
Upserted 1 points | Total in collection: 5

Idempotent upsert: before=5, after=5
```

The collection still contains **5 Points**, confirming that the existing Point was updated instead of creating a duplicate.

The final retrieval verifies the update:

```text
Retrieved point ID=3267135047:
  text     : UPDATED: Premature withdrawal penalty is 1 percent.
  doc_type : policy
  source   : FD_Policy.json
```

This confirms that the old content was successfully replaced while keeping the same Point ID.

---

### Understanding the Code Output

**TensorFlow Warning**

```text
The name tf.losses.sparse_softmax_cross_entropy is deprecated...
```

- This warning comes from TensorFlow/Keras, not Qdrant.
- It is caused by an internal dependency used by the Sentence Transformer library.
- It does not affect embedding generation or Qdrant operations and can be ignored for this demo.

---

**Collection Created**

```text
Created collection 'fd_knowledge_base'
```

- Confirms that the collection was created successfully.
- The collection is now ready to store Points.

---

**Payload Index Added**

```text
Added payload index on 'doc_type'
```

- Confirms that an index was created for the `doc_type` payload field.
- This index speeds up metadata filtering in a production Qdrant server.

---

**Documents Upserted**

```text
Upserted 5 points | Total in collection: 5
```

- Five documents were converted into embeddings and stored as five Points.
- Since all IDs were new, five new Points were inserted.

---

**Retrieve by ID**

```text
Retrieved point ID=3267135047:
```

- Qdrant directly fetched the Point using its deterministic ID.
- No embedding generation or semantic search was required.

```text
text     : Premature withdrawal incurs a 1 percent penalty...
doc_type : policy
source   : FD_Policy.json
```

- The stored payload is returned.
- The embedding vector is not shown because `with_vectors=False`.

---

**Scroll All Points**

```text
Scroll (all): 5 points
```

- All five stored Points are returned.
- Since no filter was applied, every document in the collection is listed.

---

**Scroll with Payload Filter**

```text
Scroll (doc_type='faq'): 2 points
```

- Only Points with `doc_type="faq"` are returned.
- Policy and Product documents are excluded because of the payload filter.

---

**Idempotent Upsert**

```text
Upserted 1 points | Total in collection: 5
```

- One document was upserted.
- The total Point count remains **5** because an existing Point was updated instead of inserting a new one.

```text
Idempotent upsert: before=5, after=5
```

- Confirms that no duplicate Point was created.
- The deterministic ID allowed Qdrant to replace the existing Point.

---

**Updated Point Retrieved**

```text
Retrieved point ID=3267135047:
```

```text
text : UPDATED: Premature withdrawal penalty is 1 percent.
```

- The retrieved text now shows the updated content.
- This confirms that the existing Point was replaced successfully while keeping the same ID.

---

**Payload Index Warning**

```text
Payload indexes have no effect in the local Qdrant.
```

- This warning appears because the demo uses **`:memory:`** mode.
- Local in-memory Qdrant does not use payload indexes for filtering.
- In a production Qdrant server, payload indexes improve filtering performance significantly.

---

In [1]:
"""
qdrant_collections_points.py
------------------------------
Demonstrates Qdrant's core data model: collections, points, payloads.

Shows:
  - Creating a collection
  - Generating deterministic point IDs (idempotent upserts)
  - Upserting points with payload mirroring the Document shape
  - Adding a payload index for filtered search performance
  - Retrieving a specific point by ID
  - Scrolling through all points without a query vector
  - Demonstrating idempotent upsert (re-upsert updates, not duplicates)

Uses :memory: mode -- no Docker required.
Install: pip install qdrant-client sentence-transformers
"""

import hashlib

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,          # cosine, dot, euclidean -- how to measure vector closeness
    VectorParams,      # vector size + distance metric config for a collection
    PointStruct,       # one point: id + vector + payload
    PayloadSchemaType, # tells Qdrant what type of payload index to create
    Filter,            # wraps filter conditions for a search or scroll
    FieldCondition,    # one condition on a payload field
    MatchValue,        # "must equal this exact value" match type
)
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = "fd_knowledge_base"
VECTOR_SIZE = 384        # must match the embedding model's output size
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# in-memory client -- no Docker, no server needed
client = QdrantClient(":memory:")

# load embedding model once -- reused for every encode() call
model = SentenceTransformer(MODEL_NAME)

# sample documents in the same shape the ingestion chapter produces
DOCUMENTS = [
    {
        "page_content": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.",
        "metadata": {"source": "FD_Policy.json", "doc_type": "policy", "page": 1, "chunk_index": 0},
    },
    {
        "page_content": "This penalty does not apply if the FD is closed due to the death of the depositor.",
        "metadata": {"source": "FD_Policy.json", "doc_type": "policy", "page": 1, "chunk_index": 1},
    },
    {
        "page_content": "Senior citizens receive an additional 0.5 percent interest on all tenures.",
        "metadata": {"source": "FD_Product_Guide.json", "doc_type": "product", "page": 2, "chunk_index": 0},
    },
    {
        "page_content": "What is the minimum amount to open an FD? The minimum deposit is Rs 25,000.",
        "metadata": {"source": "FD_FAQ.json", "doc_type": "faq", "page": 1, "chunk_index": 0},
    },
    {
        "page_content": "Can I withdraw my FD before maturity? Yes, subject to a penalty.",
        "metadata": {"source": "FD_FAQ.json", "doc_type": "faq", "page": 1, "chunk_index": 1},
    },
]


def make_point_id(source: str, chunk_index: int) -> int:
    """Deterministic integer ID from source path + chunk index.
    Same inputs always produce the same ID so re-upserting the same
    chunk updates it in place instead of creating a duplicate point."""
    key = f"{source}::{chunk_index}"
    # take first 8 hex chars of md5 and convert to int -- small enough for Qdrant's uint64 ID
    return int(hashlib.md5(key.encode()).hexdigest()[:8], 16)


def create_collection() -> None:
    # check existing collections to avoid crashing on a duplicate name
    existing = [c.name for c in client.get_collections().collections]

    # delete the old collection if present -- clean slate for this demo
    if COLLECTION_NAME in existing:
        client.delete_collection(COLLECTION_NAME)

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,          # every point's vector must be exactly this many floats
            distance=Distance.COSINE,  # cosine similarity -- correct for normalized vectors
        ),
    )
    print(f"Created collection '{COLLECTION_NAME}'")


def add_payload_index() -> None:
    """Adds a keyword payload index on 'doc_type'.
    Without this, a filter on doc_type scans every payload record.
    With this, Qdrant resolves the filter via the index -- much faster at scale."""
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name="doc_type",                  # the payload field to index
        field_schema=PayloadSchemaType.KEYWORD,  # keyword index = exact string match
    )
    print("Added payload index on 'doc_type'")


def upsert_documents(documents: list) -> None:
    # extract text strings for batch embedding
    texts = [d["page_content"] for d in documents]

    # embed all at once -- normalize so dot product == cosine similarity
    embeddings = model.encode(texts, normalize_embeddings=True)

    points = []
    for i, doc in enumerate(documents):
        meta = doc["metadata"]

        # deterministic ID -- same source + chunk_index always gives the same ID
        point_id = make_point_id(meta["source"], meta["chunk_index"])

        points.append(
            PointStruct(
                id=point_id,
                vector=embeddings[i].tolist(),  # numpy array -> plain list for serialization
                payload={
                    "text": doc["page_content"],    # store raw text for self-contained retrieval
                    "source": meta["source"],        # which file this chunk came from
                    "doc_type": meta["doc_type"],    # used for payload filtering
                    "page": meta["page"],            # page within the source document
                    "chunk_index": meta["chunk_index"],  # position within the page
                },
            )
        )

    # upsert = insert if new ID, replace entirely if ID already exists
    client.upsert(collection_name=COLLECTION_NAME, points=points)

    info = client.get_collection(COLLECTION_NAME)
    print(f"Upserted {len(points)} points | Total in collection: {info.points_count}")


def retrieve_by_id(source: str, chunk_index: int) -> None:
    """Fetches one specific point by its deterministic ID -- no query vector needed."""
    point_id = make_point_id(source, chunk_index)

    results = client.retrieve(
        collection_name=COLLECTION_NAME,
        ids=[point_id],
        with_payload=True,   # include the payload fields in the result
        with_vectors=False,  # skip the raw embedding -- not needed for inspection
    )

    if results:
        print(f"\nRetrieved point ID={point_id}:")
        print(f"  text     : {results[0].payload['text'][:80]}")
        print(f"  doc_type : {results[0].payload['doc_type']}")
        print(f"  source   : {results[0].payload['source']}")
    else:
        print(f"No point found for ID={point_id}")


def scroll_all(doc_type_filter: str = None) -> None:
    """Iterates through points without a query vector.
    Used for auditing, bulk re-embedding, or deletion by filter --
    not for similarity search."""

    # build a filter if a doc_type was given, otherwise scroll everything
    scroll_filter = None
    if doc_type_filter:
        scroll_filter = Filter(
            must=[
                FieldCondition(
                    key="doc_type",
                    match=MatchValue(value=doc_type_filter),
                )
            ]
        )

    # scroll returns (results, next_page_offset) -- we only need the results here
    results, _ = client.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter=scroll_filter,
        with_payload=True,
        with_vectors=False,
        limit=100,  # max points to return in one scroll call
    )

    label = f"doc_type='{doc_type_filter}'" if doc_type_filter else "all"
    print(f"\nScroll ({label}): {len(results)} points")
    for r in results:
        print(f"  [{r.payload['doc_type']}] {r.payload['text'][:60]}")


def demonstrate_idempotent_upsert() -> None:
    """Re-upserts a point that already exists (same deterministic ID).
    Point count stays the same -- it updated in place, not duplicated."""

    # record the count before re-upserting
    count_before = client.get_collection(COLLECTION_NAME).points_count

    # same source + chunk_index as the first document -- same ID will be generated
    updated_doc = {
        "page_content": "UPDATED: Premature withdrawal penalty is 1 percent.",
        "metadata": {
            "source": "FD_Policy.json",
            "doc_type": "policy",
            "page": 1,
            "chunk_index": 0,  # same chunk_index -> same ID -> in-place update
        },
    }
    upsert_documents([updated_doc])

    count_after = client.get_collection(COLLECTION_NAME).points_count

    print(f"\nIdempotent upsert: before={count_before}, after={count_after}")
    print("  Count unchanged -- updated in place, no duplicate created")

    # confirm the content was actually updated
    retrieve_by_id("FD_Policy.json", chunk_index=0)


# ----------------------------------------------------------------------
# Run all demos in order
# ----------------------------------------------------------------------

# step 1: create collection
create_collection()

# step 2: add payload index BEFORE upserting so Qdrant indexes as points arrive
add_payload_index()

# step 3: embed and upsert all sample documents
upsert_documents(DOCUMENTS)

# step 4: fetch one specific point by its deterministic ID
retrieve_by_id("FD_Policy.json", chunk_index=0)

# step 5: scroll through all points (no query vector)
scroll_all()

# step 6: scroll only FAQ points using the payload index
scroll_all(doc_type_filter="faq")

# step 7: re-upsert one point and show the count stays the same
demonstrate_idempotent_upsert()


Created collection 'fd_knowledge_base'
Added payload index on 'doc_type'
Upserted 5 points | Total in collection: 5

Retrieved point ID=3267135047:
  text     : Premature withdrawal incurs a 1 percent penalty on the applicable rate.
  doc_type : policy
  source   : FD_Policy.json

Scroll (all): 5 points
  [faq] What is the minimum amount to open an FD? The minimum deposi
  [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
  [product] Senior citizens receive an additional 0.5 percent interest o
  [policy] This penalty does not apply if the FD is closed due to the d
  [policy] Premature withdrawal incurs a 1 percent penalty on the appli

Scroll (doc_type='faq'): 2 points
  [faq] What is the minimum amount to open an FD? The minimum deposi
  [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
Upserted 1 points | Total in collection: 5

Idempotent upsert: before=5, after=5
  Count unchanged -- updated in place, no duplicate created

Retrieved point ID=32671

C:\Users\pauls\AppData\Local\Temp\ipykernel_42232\3686356635.py:99: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(
